# V3 Name That LLM — Prompt Joint Attribution

Based on the frozen V2 style-fusion model; infers prompt groups only from **unlabeled test texts**, then assigns model labels within groups using the Hungarian algorithm. Incomplete groups shorter than 16 use independent predictions. Kaggle hidden-test distribution may differ from the grouped validation protocol.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

def locate(name):
    candidates=[Path("/kaggle/input/competitions/name-that-llm")/name,
                Path("/kaggle/input/name-that-llm")/name]
    for p in candidates:
        if p.is_file(): return p
    raise FileNotFoundError("Attach official Name That LLM competition data: "+name)

train=pd.read_csv(locate("train.csv"))
test=pd.read_csv(locate("test.csv"))
assert list(train.columns)==["id","prompt_id","text","model"]
assert list(test.columns)==["id","text"]
assert train.model.nunique()==18 and train.id.is_unique
assert test.id.is_unique
assert train[["id","prompt_id","text","model"]].notna().all().all()
assert test[["id","text"]].notna().all().all()
print("train",train.shape,"test",test.shape)


In [ ]:
TOKENS = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?", re.UNICODE)
SENTENCES = re.compile(r"[.!?]+(?:\s|$)")
WORD_SET = [
    "the", "and", "of", "to", "in", "is", "that", "this", "which",
    "with", "as", "by", "for", "on", "from", "are", "can", "may", "might",
    "will", "should", "could", "we", "you", "i", "they", "it", "not",
    "but", "or", "a", "an", "their", "its", "our", "your",
    "however", "therefore", "indeed", "perhaps", "ultimately",
    "although", "while", "because", "rather", "instead", "through",
    "each", "every", "even", "yet", "only", "often", "also",
]
PHRASES = [
    "for example", "such as", "in fact", "rather than",
    "not only", "not just", "on the other hand", "in other words",
    "at the same time", "in this way", "as a result",
]
PUNCT = list(",;:.!?-—–()[]{}\"'“”‘’/\\*#\n")


def style_vector(text: str) -> np.ndarray:
    """Surface-only, topic-agnostic style statistics."""
    words = [w.lower() for w in TOKENS.findall(text)]
    n = max(1, len(words))
    chars = max(1, len(text))
    lower = text.lower()
    count = {}
    for w in words:
        count[w] = count.get(w, 0) + 1
    counts = np.asarray(list(count.values()), dtype=float) if count else np.array([0.0])
    sent_len = [len(TOKENS.findall(s)) for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    if not sent_len:
        sent_len = [n]
    words_len = [len(w) for w in words] or [0]
    vals = [
        np.log1p(chars), np.log1p(n),
        float(np.mean(words_len)), float(np.std(words_len)),
        np.mean(sent_len), np.std(sent_len),
        len(count) / n, np.sum(counts == 1) / n,
        np.sum(counts >= 3) / n,
        sum(len(w) >= 8 for w in words) / n,
        sum(len(w) <= 3 for w in words) / n,
        sum(w[0].isupper() for w in TOKENS.findall(text)) / n,
        sum(c.isupper() for c in text) * 100 / chars,
        sum(c.isdigit() for c in text) * 100 / chars,
        sum(ord(c) > 127 for c in text) * 100 / chars,
        sum(c.isspace() for c in text) * 100 / chars,
        text.count("\n\n") * 100 / chars,
        len(re.findall(r"\b\d+(?:\.\d+)?\b", text)) / n,
    ]
    vals.extend([text.count(p) * 100 / chars for p in PUNCT])
    vals.extend([count.get(w, 0) * 100 / n for w in WORD_SET])
    vals.extend([lower.count(p) * 100 / n for p in PHRASES])
    vals.append(sum(text.startswith(prefix) for prefix in (
        "In ", "The ", "A ", "When ", "Imagine ", "Consider ", "There ", "Every ",
    )))
    return np.asarray(vals, dtype=np.float32)


def style_matrix(texts) -> np.ndarray:
    return np.asarray([style_vector(text) for text in texts], dtype=np.float32)


In [ ]:
def group_greedy_cosine(texts:list[str], group_size:int=18):
    v=TfidfVectorizer(analyzer="word",ngram_range=(1,2),stop_words="english",
                      min_df=1,max_features=50000,sublinear_tf=True)
    x=v.fit_transform(texts)
    sim=cosine_similarity(x)
    n=len(texts)
    ii,jj=np.triu_indices(n,k=1)
    order=np.argsort(-sim[ii,jj],kind="stable")
    parent=list(range(n))
    sizes=[1]*n
    num_groups=n
    def root(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]
            x=parent[x]
        return x
    for pos in order:
        i,j=int(ii[pos]),int(jj[pos])
        ri,rj=root(i),root(j)
        if ri==rj:continue
        if sizes[ri]+sizes[rj]>group_size:continue
        if sizes[ri]<sizes[rj]:ri,rj=rj,ri
        parent[rj]=ri
        sizes[ri]+=sizes[rj]
        num_groups-=1
        if num_groups==int(np.ceil(n/group_size)):
            break
    roots=[root(i) for i in range(n)]
    _, groups=np.unique(roots,return_inverse=True)
    return groups


In [ ]:
def predict_joint(margins,groups,model_names,min_group_size=16):
    """Allocate distinct model labels within near-complete prompt groups."""
    if len(margins)!=len(groups) or margins.shape[1]!=len(model_names):
        raise ValueError("Mismatched model/group dimensions")
    independent=np.asarray(model_names)[margins.argmax(axis=1)]
    matched=independent.copy()
    applied=0
    for cluster in np.unique(groups):
        rows=np.flatnonzero(groups==cluster)
        if len(rows)<min_group_size or len(rows)>len(model_names):
            continue
        ridx,cidx=linear_sum_assignment(-margins[rows])
        matched[rows[ridx]]=model_names[cidx]
        applied+=1
    return independent,matched,applied

def train_v2_margins(xtr,xva,ytr):
    """Exactly the frozen V2 best: char+word+style LinearSVC."""
    char=TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,
                         max_features=100000,sublinear_tf=True,dtype=np.float32)
    word=TfidfVectorizer(analyzer="word",ngram_range=(1,2),min_df=2,
                         max_features=80000,sublinear_tf=True,dtype=np.float32)
    ct,cv=char.fit_transform(xtr),char.transform(xva)
    wt,wv=word.fit_transform(xtr),word.transform(xva)
    scale=StandardScaler()
    st=csr_matrix(np.clip(scale.fit_transform(style_matrix(xtr)),-4,4))
    sv=csr_matrix(np.clip(scale.transform(style_matrix(xva)),-4,4))
    model=LinearSVC(C=1.0,max_iter=4000)
    model.fit(hstack([ct,wt,st*0.08],format="csr"),ytr)
    return model.decision_function(hstack([cv,wv,sv*0.08],format="csr")),model.classes_


In [ ]:
margins, classes = train_v2_margins(
    train.text.tolist(), test.text.tolist(), train.model.to_numpy()
)
if len(test) < 16:
    # Kaggle interactive session exposes 3 example rows only.
    predictions = classes[margins.argmax(axis=1)]
    print("Small interactive test batch: independent fallback.")
else:
    groups = group_greedy_cosine(test.text.tolist(), group_size=18)
    _, predictions, applied = predict_joint(
        margins, groups, classes, min_group_size=16
    )
    print("Joint assignment applied to", applied,
          "near-complete inferred prompt groups.")

submission = pd.DataFrame({"id": test.id, "model": predictions})
assert submission.id.equals(test.id)
assert submission.id.is_unique
assert submission.model.isin(train.model.unique()).all()
assert submission.notna().all().all()
out = Path("/kaggle/working/submission.parquet") if Path("/kaggle/working").is_dir() else Path("submission.parquet")
submission.to_parquet(out, index=False)
print("Submitted",len(submission),"rows to",out)
print(submission.head())
